[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/07_position_encoding.ipynb)

# 07. Position encoding — absolute PE to real RoPE rotations

이전 RoPE 코드는 rotation angle을 단순 `position × [1,2,...]`로 만들었고, 2D/3D section은 좌표만 출력해서 실제 RoPE를 적용하지 않았다.

이번 버전은 **RoFormer의 inverse-frequency rotation을 Q/K에 적용**하고, 2D/3D에서는 hidden dimensions를 spatial axes에 나눠 axial RoPE를 실제로 수행한다.


In [ ]:
import math

import torch
import torch.nn as nn

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Learned absolute position embedding


In [ ]:
sequence_length = 5
hidden_dim = 8

tokens = torch.zeros(
    1, sequence_length, hidden_dim,
    device=device,
)
position_embedding = nn.Embedding(
    sequence_length,
    hidden_dim,
).to(device)
positions = torch.arange(sequence_length, device=device)

positioned_tokens = tokens + position_embedding(positions)[None]
print("absolute-positioned tokens:", positioned_tokens.shape)


## 2. Sinusoidal absolute encoding


In [ ]:
def sinusoidal_absolute_encoding(length, dim, device):
    position = torch.arange(length, device=device).float()[:, None]
    inverse_frequency = torch.exp(
        torch.arange(0, dim, 2, device=device).float()
        * (-math.log(10000.0) / dim)
    )

    angles = position * inverse_frequency[None]
    encoding = torch.zeros(length, dim, device=device)
    encoding[:, 0::2] = angles.sin()
    encoding[:, 1::2] = angles.cos()
    return encoding


sinusoidal = sinusoidal_absolute_encoding(
    sequence_length,
    hidden_dim,
    device,
)
print(sinusoidal)


## 3. RoPE: rotate Q and K with geometric inverse frequencies

RoPE는 position vector를 token embedding에 더하지 않는다. query/key channel pairs를 position-dependent angle로 회전시킨다. frequency는 `theta_i = 10000^{-2i/d}` 계열의 geometric scale을 사용한다.


In [ ]:
def rope_angles(positions, dim, base=10000.0):
    assert dim % 2 == 0

    pair_index = torch.arange(
        0, dim, 2,
        device=positions.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (
        base ** (pair_index / dim)
    )
    return positions.float()[:, None] * inverse_frequency[None]


def apply_rope(x, positions):
    dim = x.size(-1)
    angles = rope_angles(positions, dim)

    cos = angles.cos()
    sin = angles.sin()

    even = x[..., 0::2]
    odd = x[..., 1::2]

    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos

    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


q = torch.randn(1, sequence_length, hidden_dim, device=device)
k = torch.randn_like(q)
positions = torch.arange(sequence_length, device=device)

q_rope = apply_rope(q, positions)
k_rope = apply_rope(k, positions)

print("Q before/after:", q.shape, q_rope.shape)
print("K before/after:", k.shape, k_rope.shape)


## 4. Relative-position effect in the QK dot product

같은 content vector를 서로 다른 absolute position에서 회전시키면 QK similarity가 두 position의 **차이**에 따라 변한다. 이것이 RoPE가 relative position을 attention score에 넣는 핵심 성질이다.


In [ ]:
content = torch.tensor(
    [[1.0, 0.5, -0.3, 0.7, 0.2, -0.4, 0.8, 0.1]],
    device=device,
)

content_sequence = content.expand(4, -1).unsqueeze(0)
position_ids = torch.arange(4, device=device)
rotated = apply_rope(content_sequence, position_ids)

similarity = rotated[0] @ rotated[0].T
print("RoPE QK similarity:\n", similarity)


## 5. Axial 2D RoPE

2D vision token에서는 channel 일부를 x-axis RoPE, 나머지를 y-axis RoPE에 배정할 수 있다. 각 axis는 자기 coordinate만 사용해 회전한다.


In [ ]:
def axial_rope_2d(x, height, width):
    assert x.size(-1) % 2 == 0

    half_dim = x.size(-1) // 2
    assert half_dim % 2 == 0

    grid_y, grid_x = torch.meshgrid(
        torch.arange(height, device=x.device),
        torch.arange(width, device=x.device),
        indexing="ij",
    )

    x_part, y_part = x.split(half_dim, dim=-1)
    x_rotated = apply_rope(x_part, grid_x.reshape(-1))
    y_rotated = apply_rope(y_part, grid_y.reshape(-1))

    return torch.cat([x_rotated, y_rotated], dim=-1)


vision_tokens = torch.randn(1, 2 * 3, 16, device=device)
vision_rope = axial_rope_2d(vision_tokens, height=2, width=3)

print("2D axial RoPE:", vision_rope.shape)


## 6. Axial 3D RoPE

video/3D transformer에서는 channel을 x/y/time 또는 x/y/z axes에 나눌 수 있다. 실제 모델마다 axis channel ratio가 다를 수 있다. 아래는 이해하기 쉬운 균등 3-way split이다.


In [ ]:
def axial_rope_3d(x, depth, height, width):
    assert x.size(-1) % 3 == 0

    axis_dim = x.size(-1) // 3
    assert axis_dim % 2 == 0

    grid_z, grid_y, grid_x = torch.meshgrid(
        torch.arange(depth, device=x.device),
        torch.arange(height, device=x.device),
        torch.arange(width, device=x.device),
        indexing="ij",
    )

    x_part, y_part, z_part = x.split(axis_dim, dim=-1)

    x_rotated = apply_rope(x_part, grid_x.reshape(-1))
    y_rotated = apply_rope(y_part, grid_y.reshape(-1))
    z_rotated = apply_rope(z_part, grid_z.reshape(-1))

    return torch.cat(
        [x_rotated, y_rotated, z_rotated],
        dim=-1,
    )


volume_tokens = torch.randn(1, 2 * 2 * 2, 24, device=device)
volume_rope = axial_rope_3d(
    volume_tokens,
    depth=2,
    height=2,
    width=2,
)

print("3D axial RoPE:", volume_rope.shape)


## References and provenance

**Sinusoidal PE** — Vaswani et al. geometric sinusoidal absolute encoding을 반영했다.

**RoPE** — Su et al., *RoFormer*. Q/K pair rotation과 geometric inverse frequencies를 반영했다.

**2D/3D RoPE** — vision/video transformer implementations에서 spatial/temporal axes에 channel subspaces를 배정하는 axial extension을 반영했다. CogVideoX 등 실제 모델은 axis별 channel 비율을 서로 다르게 둘 수 있다.
